# Coupled tumor · oxygen · necrosis

The **90-day coupled example** with a matched **Fisher–KPP reference** on the same 2 mm grid. **Side by side** shows all four fields in columns. The rows show axial, coronal and sagittal planes at the same time and RAS slice position.

Use the shared **Frame** / playback controls and **View x, y, z** sliders. **Planes** can focus on one orientation. **Display → Single field** returns to the original view. Expand **Simulation parameters & export** to adjust the model or export an individual field. Parameters are illustrative.

Choose **Run All** to create the interactive view. See the [recorded day-90 comparison](outputs/coupled_example/day90_zoom_axial.png), [20-second movie](outputs/coupled_example/comparison_slow_20s.mp4), and [scientific paper](paper/output/pdf/brainweb_fisher_kpp_manuscript.pdf) for results without running the notebook.

In [ ]:
%matplotlib widget
import json
from IPython.display import display
from brainweb_kpp.data import ROOT
import importlib
import brainweb_kpp.ui as explorer_ui
importlib.reload(explorer_ui)
TumorExplorer = explorer_ui.TumorExplorer
previous_app = globals().get("app")

app = TumorExplorer()
example = json.loads((ROOT / "examples/coupled.json").read_text())
app._busy = True
try:
    for name, value in example.items():
        if name in ("origin_mm", "radii_mm"):
            prefix = "origin_" if name == "origin_mm" else "radius_"
            for axis, coordinate in zip("xyz", value):
                app.controls[prefix + axis].value = coordinate
        else:
            app.controls[name].value = value
    app.dx.value = 2.0
    app.compare.value = True
    app.store.value = True
    app.outpath.value = "outputs/coupled_example"
finally:
    app._busy = False
app._model_controls()
app.tabs.selected_index = 2
if (previous_app is not None and previous_app.result is not None
        and previous_app.parameters() == app.parameters()
        and previous_app.brain.spacing == app.dx.value
        and previous_app.result.volumes is not None
        and previous_app.result.comparison is not None):
    # Reuse the completed simulation when refreshing the viewer.
    app.brain = previous_app.brain
    app.result = previous_app.result
    app.result_view.options = [("Viable tumor", "viable"), ("Oxygen availability", "oxygen"),
                               ("Necrotic material", "necrosis"), ("Fisher–KPP reference", "reference")]
    app.result_view.disabled = app.display_layout.disabled = False
    app.preview_mode.disabled = True
    app.time.max = app.play.max = len(app.result.times) - 1
    app.time.value = app.time.max
    app.play.disabled = False
    app.export_button.disabled = False
    app.progress.value = 100
    for slider, coordinate in zip(app.view, app.parameters().origin_mm):
        slider.value = round(coordinate)
    app.status.value = "<b>Completed run loaded:</b> 90 days · 31 frames · matched Fisher–KPP reference · saved 3D slices."
else:
    app._run()
assert app.result is not None, app.status.value
assert app.result.comparison is not None
assert app.result.times[-1] == 90
assert len(app.result_view.options) == 4
app.display_layout.value = "comparison"
app.settings.selected_index = None
if previous_app is not None:
    import matplotlib.pyplot as plt
    plt.close(previous_app.fig)
preview_folder = ROOT / "outputs/coupled_example"
preview_folder.mkdir(parents=True, exist_ok=True)
(preview_folder / "side_by_side.png").write_bytes(app.comparison_image.value)
display(app.widget)
